# Thể hiện xu hướng sử dụng promo của công ty
file: promotions.csv + other_items.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. ĐỌC DỮ LIỆU
df_promos = pd.read_csv('../dataset/promotions.csv')
df_orders = pd.read_csv('../dataset/order_items.csv')
df_products = pd.read_csv('../dataset/products.csv')

# 2. XỬ LÝ DỮ LIỆU
# --- Tính Doanh thu ---
df_orders['revenue'] = (df_orders['quantity'] * df_orders['unit_price']) - df_orders['discount_amount']

# --- Xử lý Promo ---
df_orders['promo_id'] = df_orders['promo_id'].astype(str)
df_promos['promo_id'] = df_promos['promo_id'].astype(str)

# Lấy tên khuyến mãi gốc (cắt bỏ năm)
df_promos['base_promo_name'] = df_promos['promo_name'].str.replace(r'\s*\d{4}', '', regex=True).str.strip()

# --- Kết nối Dữ liệu ---
df_merged = pd.merge(df_orders, df_promos[['promo_id', 'base_promo_name']], on='promo_id', how='left')
df_merged['base_promo_name'] = df_merged['base_promo_name'].fillna('No Promo')
df_merged['base_promo_name'] = df_merged['base_promo_name'].replace(['', 'nan'], 'No Promo')

# (Lưu ý: Không bắt buộc phải merge với products.csv nữa nếu biểu đồ 2 không dùng đến category, 
# nhưng tôi vẫn giữ lại ở đây để đảm bảo logic luồng dữ liệu của bạn không bị phá vỡ)
df_merged = pd.merge(df_merged, df_products[['product_id', 'category']], on='product_id', how='left')

# --- Xử lý dữ liệu Lịch trình TẦN SUẤT theo Tháng ---
df_promos['start_date'] = pd.to_datetime(df_promos['start_date'])
df_promos['end_date'] = pd.to_datetime(df_promos['end_date'])
promo_month_years = []
for _, row in df_promos.dropna(subset=['start_date', 'end_date']).iterrows():
    dates = pd.date_range(start=row['start_date'], end=row['end_date'])
    unique_year_months = dates.to_period('M').unique()
    for ym in unique_year_months:
        promo_month_years.append({
            'base_promo_name': row['base_promo_name'], 
            'year': ym.year,
            'month': ym.month
        })

df_pmy = pd.DataFrame(promo_month_years).drop_duplicates()
df_freq = df_pmy.groupby(['base_promo_name', 'month'])['year'].count().reset_index(name='frequency')

pivot_freq = df_freq.pivot(index='base_promo_name', columns='month', values='frequency').fillna(0)
pivot_freq = pivot_freq.reindex(columns=range(1, 13), fill_value=0)
pivot_freq.columns = [f'Tháng {i}' for i in range(1, 13)]


# 3. TRỰC QUAN HÓA (VISUALIZATION)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(15, 16))

# =====================================================================
# BIỂU ĐỒ 1: BẢN ĐỒ NHIỆT THỂ HIỆN TẦN SUẤT XUẤT HIỆN CỦA PROMO
# =====================================================================
ax1 = axes[0]

sns.heatmap(
    pivot_freq, 
    cmap='YlOrBr', 
    annot=True,     
    fmt='g',        
    linewidths=1.5, 
    linecolor='white', 
    ax=ax1
)
ax1.set_title('Tần Suất Xuất Hiện Của Các Khuyến Mãi Theo Tháng (Số năm hoạt động)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Tháng', fontsize=12)
ax1.set_ylabel('Chương trình Khuyến mãi', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# =====================================================================
# BIỂU ĐỒ 2: SO SÁNH TỔNG DOANH THU GIỮA CÁC CHƯƠNG TRÌNH KHUYẾN MÃI
# =====================================================================
ax2 = axes[1]

# Gom nhóm tính tổng doanh thu theo từng tên chương trình khuyến mãi
promo_comparison = df_merged.groupby('base_promo_name')['revenue'].sum().reset_index()

# Sắp xếp giảm dần để dễ so sánh
promo_comparison = promo_comparison.sort_values('revenue', ascending=False)

sns.barplot(
    data=promo_comparison, 
    x='base_promo_name', 
    y='revenue', 
    palette='viridis', 
    ax=ax2
)

# Hiển thị số liệu doanh thu trực tiếp trên đầu mỗi cột
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():,.0f}", 
                 (p.get_x() + p.get_width() / 2., p.get_height()), 
                 ha='center', va='bottom', 
                 fontsize=10, color='black', xytext=(0, 5), 
                 textcoords='offset points')

ax2.set_title('So Sánh Tổng Doanh Thu (Revenue) Giữa Các Chương Trình Khuyến Mãi', fontsize=14, fontweight='bold')
ax2.set_xlabel('Chương Trình Khuyến Mãi')

# Thể hiện quan hệ giữa promo với sales

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC DỮ LIỆU TỪ ../dataset/
# ==========================================
df_promos = pd.read_csv('../dataset/promotions.csv')
df_order_items = pd.read_csv('../dataset/order_items.csv')
df_products = pd.read_csv('../dataset/products.csv')

# ==========================================
# 2. XỬ LÝ DỮ LIỆU 
# ==========================================
# Tính doanh thu từng dòng
df_order_items['revenue'] = (df_order_items['quantity'] * df_order_items['unit_price']) - df_order_items['discount_amount']

# Xử lý ID và Tên Promo (gộp năm)
df_order_items['promo_id'] = df_order_items['promo_id'].astype(str).replace(['nan', 'None'], 'No Promo')
df_promos['promo_id'] = df_promos['promo_id'].astype(str)
df_promos['base_promo_name'] = df_promos['promo_name'].str.replace(r'\s*\d{4}', '', regex=True).str.strip()

# Ghép bảng để lấy thông tin Loại mặt hàng (Category) và Tên Promo
df_merged = pd.merge(df_order_items, df_products[['product_id', 'category']], on='product_id', how='left')
df_merged = pd.merge(df_merged, df_promos[['promo_id', 'base_promo_name']], on='promo_id', how='left')
df_merged['base_promo_name'] = df_merged['base_promo_name'].fillna('No Promo')

# --- TÍNH TOÁN TOTAL & RATE ---
# 2.1. Tính tổng doanh thu theo (Loại mặt hàng + Promo)
cat_promo_stats = df_merged.groupby(['category', 'base_promo_name'])['revenue'].sum().reset_index()

# 2.2. Tính tổng doanh thu của MỖI LOẠI MẶT HÀNG (Dùng làm mẫu số tính Rate)
cat_totals = df_merged.groupby('category')['revenue'].sum().reset_index(name='category_total_revenue')

# 2.3. Ghép lại để tính Rate (% đóng góp của Promo vào mặt hàng đó)
cat_promo_stats = pd.merge(cat_promo_stats, cat_totals, on='category', how='left')
cat_promo_stats['contribution_rate'] = (cat_promo_stats['revenue'] / cat_promo_stats['category_total_revenue']) * 100

# Lọc Top 10 Loại mặt hàng có doanh thu cao nhất để biểu đồ không bị rối mắt
top_categories = cat_totals.sort_values('category_total_revenue', ascending=False).head(10)['category']
cat_promo_stats_top = cat_promo_stats[cat_promo_stats['category'].isin(top_categories)]

# Tạo bảng Pivot (Ma trận) cho Heatmap tỷ lệ Rate
pivot_rate_top = cat_promo_stats_top.pivot(index='category', columns='base_promo_name', values='contribution_rate').fillna(0)

# ==========================================
# 3. TRỰC QUAN HÓA (VISUALIZATION)
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(16, 18))

# -------------------------------------------------------------------
# BIỂU ĐỒ 1: TỔNG DOANH THU THEO LOẠI MẶT HÀNG VÀ PROMO (TOTAL)
# -------------------------------------------------------------------
ax1 = axes[0]
sns.barplot(
    data=cat_promo_stats_top, 
    x='category', 
    y='revenue', 
    hue='base_promo_name', 
    palette='tab20', 
    ax=ax1
)
ax1.set_title('TỔNG DOANH THU: Sự Tác Động Của Promo Lên Từng Loại Mặt Hàng (Top 10 Category)', fontsize=15, fontweight='bold', pad=15)
ax1.set_ylabel('Tổng Doanh Thu (VND)', fontsize=12)
ax1.set_xlabel('Loại Mặt Hàng (Category)', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.legend(title='Tên Khuyến Mãi', bbox_to_anchor=(1.01, 1), loc='upper left')

# -------------------------------------------------------------------
# BIỂU ĐỒ 2: TỶ LỆ ĐÓNG GÓP CỦA PROMO VÀO DOANH THU MẶT HÀNG (RATE %)
# -------------------------------------------------------------------
ax2 = axes[1]
# Dùng heatmap màu xanh, hiển thị số kèm dấu %
sns.heatmap(
    pivot_rate_top, 
    annot=True, 
    fmt='.1f', 
    cmap='Blues', 
    linewidths=1, 
    linecolor='white',
    cbar_kws={'label': 'Tỷ lệ đóng góp (%)'},
    ax=ax2
)

# Thêm ký hiệu % vào các con số trong Heatmap cho trực quan
for t in ax2.texts: t.set_text(t.get_text() + " %")

ax2.set_title('TỶ LỆ (RATE): Promo Đóng Góp Bao Nhiêu % Vào Doanh Thu Của Mặt Hàng Đó?', fontsize=15, fontweight='bold', pad=15)
ax2.set_ylabel('Loại Mặt Hàng (Category)', fontsize=12)
ax2.set_xlabel('Tên Khuyến Mãi', fontsize=12)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Thể hiện quan hệ giữa rev/cogs và các promo


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================
df_promos = pd.read_csv('../dataset/promotions.csv')
df_order_items = pd.read_csv('../dataset/order_items.csv')
# Cần products.csv để lấy giá vốn (cogs) tính Lợi nhuận
df_products = pd.read_csv('../dataset/products.csv')

# ==========================================
# 2. XỬ LÝ DỮ LIỆU
# ==========================================
# Lấy ra thông tin 'Năm' từ cột start_date của chiến dịch
df_promos['start_date'] = pd.to_datetime(df_promos['start_date'])
df_promos['year'] = df_promos['start_date'].dt.year

# Xử lý chuẩn hóa ID để merge
df_order_items['promo_id'] = df_order_items['promo_id'].astype(str)
df_promos['promo_id'] = df_promos['promo_id'].astype(str)

# Ghép dữ liệu để lấy Tên Promo, Năm diễn ra và Giá vốn (cogs)
df_merged = pd.merge(df_order_items, df_promos[['promo_id', 'promo_name', 'year']], on='promo_id', how='inner')
df_merged = pd.merge(df_merged, df_products[['product_id', 'cogs']], on='product_id', how='left')

# Tính toán Doanh thu và Lợi nhuận
# Doanh thu = (Số lượng * Đơn giá) - Giảm giá
df_merged['revenue'] = (df_merged['quantity'] * df_merged['unit_price']) - df_merged['discount_amount']
# Giá vốn tổng = Số lượng * Giá vốn từng SP
df_merged['total_cogs'] = df_merged['quantity'] * df_merged['cogs']
# Lợi nhuận = Doanh thu - Giá vốn tổng
df_merged['profit'] = df_merged['revenue'] - df_merged['total_cogs']

# Gom nhóm tổng số liệu theo Tên chiến dịch và Năm
promo_stats = df_merged.groupby(['promo_name', 'year'])[['revenue', 'profit']].sum().reset_index()

# YÊU CẦU CỐT LÕI: SẮP XẾP THEO NĂM
# Sắp xếp giảm dần theo năm (năm mới nhất ở trên cùng), nếu cùng năm thì xếp theo tên theo alphabet
promo_stats = promo_stats.sort_values(by=['year', 'promo_name'], ascending=[False, True]).reset_index(drop=True)


# ==========================================
# 3. TRỰC QUAN HÓA (VISUALIZATION)
# ==========================================
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(16, 14))

# Cột Doanh thu (Màu xanh, hướng sang phải)
ax.barh(
    y=promo_stats['promo_name'], 
    width=promo_stats['revenue'], 
    color='#7cb5ec', 
    label='Doanh thu (Revenue)', 
    height=0.65
)

# Cột Lợi nhuận (Màu đỏ/hồng, các giá trị âm sẽ tự động lùi về bên trái trục 0)
ax.barh(
    y=promo_stats['promo_name'], 
    width=promo_stats['profit'], 
    color='#f15c80', 
    label='Lợi nhuận (Profit)', 
    alpha=0.8, 
    height=0.65
)

# Vẽ một đường ranh giới đen đứt nét tại mốc 0 (Zero-line) để dễ quan sát lỗ/lãi
ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5)

# Căn chỉnh tiêu đề và nhãn
ax.set_title('PHÂN TÍCH TOÀN BỘ CHIẾN DỊCH KHUYẾN MÃI: DOANH THU VS LỢI NHUẬN\n(Đã sắp xếp theo Năm diễn ra)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Giá trị (VND)', fontsize=12)
ax.set_ylabel('Tên Chiến dịch', fontsize=12)

# Hiển thị chú thích
ax.legend(loc='upper right', frameon=True)

# Lật ngược trục Y để các dòng đầu tiên (năm mới nhất) nằm ở trên cùng của biểu đồ
ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Hiệu quả tài chính của Fall Launch

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. ĐỌC VÀ LỌC DỮ LIỆU CHIẾN DỊCH FALL LAUNCH
# ==========================================
df_promos = pd.read_csv('../dataset/promotions.csv')
df_order_items = pd.read_csv('../dataset/order_items.csv')
df_products = pd.read_csv('../dataset/products.csv')

# Lấy ID của các chiến dịch Fall Launch
fall_launch_ids = df_promos[df_promos['promo_name'].str.contains('Fall Launch', na=False)]['promo_id'].unique()

# Lọc các đơn hàng áp dụng Fall Launch
df_fall = df_order_items[df_order_items['promo_id'].isin(fall_launch_ids)].copy()

# Ghép với bảng sản phẩm để lấy COGS
df_fall = df_fall.merge(df_products[['product_id', 'product_name', 'category', 'cogs']], on='product_id', how='left')

# ==========================================
# 2. TÍNH TOÁN CÁC CHỈ SỐ THUA LỖ
# ==========================================
# Doanh thu thuần = (Qty * Price) - Discount
df_fall['net_revenue'] = (df_fall['quantity'] * df_fall['unit_price']) - df_fall['discount_amount']
# Tổng vốn = Qty * COGS
df_fall['total_cogs'] = df_fall['quantity'] * df_fall['cogs']
# Lợi nhuận = Doanh thu thuần - Tổng vốn
df_fall['profit'] = df_fall['net_revenue'] - df_fall['total_cogs']

# ==========================================
# 3. TRỰC QUAN HÓA TÌM "KEYS" THUA LỖ
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# --- Biểu đồ 1: Cấu trúc Doanh thu vs Chi phí (Phân tích vì sao lỗ) ---
fall_totals = df_fall[['net_revenue', 'total_cogs', 'discount_amount', 'profit']].sum()
fall_totals_plot = pd.DataFrame({'Metric': fall_totals.index, 'Value': fall_totals.values})
sns.barplot(data=fall_totals_plot, x='Metric', y='Value', palette='vlag', ax=axes[0,0])
axes[0,0].set_title('Tổng Quan Tài Chính Chiến Dịch Fall Launch', fontsize=14, fontweight='bold')

# --- Biểu đồ 2: Lợi nhuận theo Category trong Fall Launch ---
cat_profit = df_fall.groupby('category')['profit'].sum().sort_values()
sns.barplot(x=cat_profit.values, y=cat_profit.index, palette='RdYlGn', ax=axes[0,1])
axes[0,1].set_title('Lợi Nhuận Theo Ngành Hàng (Tìm nhóm gây lỗ nhất)', fontsize=14, fontweight='bold')

# --- Biểu đồ 3: Hiệu quả theo Kênh Quảng cáo (Promo Channel) ---
# Ghép lại với promo để lấy channel
df_fall_chan = df_fall.merge(df_promos[['promo_id', 'promo_channel']], on='promo_id')
chan_eval = df_fall_chan.groupby('promo_channel')['profit'].sum().sort_values()
sns.barplot(x=chan_eval.values, y=chan_eval.index, palette='coolwarm', ax=axes[1,0])
axes[1,0].set_title('Lợi Nhuận Fall Launch Theo Kênh Marketing', fontsize=14, fontweight='bold')

# --- Biểu đồ 4: Tương quan Discount vs Profit (Mức độ "đốt tiền") ---
sns.scatterplot(data=df_fall, x='discount_amount', y='profit', alpha=0.5, color='red', ax=axes[1,1])
axes[1,1].axhline(0, color='black', linestyle='--')
axes[1,1].set_title('Tương Quan Giảm Giá vs Lợi Nhuận Từng Đơn Hàng', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# --- XUẤT CÁC "KEYS" THẤT BẠI ---
print(f"--- CHIẾN LƯỢC GIẢI MÃ THẤT BẠI THÁNG 8 (FALL LAUNCH) ---")
print(f"Key 13 (Nhóm hàng gây lỗ nặng nhất): {cat_profit.idxmin()}")
print(f"Key 14 (Kênh Marketing kém hiệu quả nhất): {chan_eval.idxmin()}")
print(f"Key 15 (Mức độ Discount): Nếu cột Discount_amount ở biểu đồ 1 cao xấp xỉ Profit, bạn đang giảm giá quá sâu.")